E-19: inference for the average marginal effect (H19-Inf, H19-Cex; PR4c `cor:ame_asymp_normality`, `rem:ame_conditions`)

Design: `doc/2026-10-07_experiment_registration.md` (parent repository), E-19, and §1.4–§1.9. One notebook for every stage; the runner passes the stage:

- `python3 notebooks/experiments/run_registered.py E-19 stage0 16_E19_ame_inference.ipynb`
- `... E-19 stage0.5 ...`, then `... E-19 stage1 ...` (each after the previous stage is committed).

DGPs, the replication function and the aggregation are in `grrexp/e19.py`; the pilot and the confirmatory stage run the same aggregation. A solver that raises stops the notebook and the run is recorded as failed (§1.1).

In [1]:
import io
import json
import os
import time

import pandas as pd

from grrexp import env, outputs

gr = env.use_submodule_genriesz()
from grrexp import baselines, e12, e19, parallel  # noqa: E402
from grrexp.seeds import CONFIRMATORY_ENTROPY, PILOT_ENTROPY, Seeds  # noqa: E402

STAGE = json.loads(os.environ[outputs.RUN_ENV])["stage"]
rec = outputs.RunRecorder(19, STAGE, cells=None if STAGE == "stage0" else e19.CELLS,
                          arms=None if STAGE == "stage0" else [a.replace("|", "-") for a in e19.ARM_LABELS])
STAGE

'stage1'

# Stage 0: population quantities, the certificate of the SQ population minimizer, and the predictions (§1.9)

In [2]:
if STAGE == "stage0":
    pop, stage0_warnings, _ = baselines.run_recording_warnings(e19.stage0)
    rec.write_stage0("e19_population", pop)
    print(json.dumps({k: pop["inference"][k] for k in ("certified", "checks", "theta0", "V_star", "c_n")}, indent=1))
    print(json.dumps(pop["counterexamples"], indent=1))
    if not pop["inference"]["certified"]:
        raise RuntimeError("the SQ population minimizer of DGP19Q is not certified")
    rec.finalize(R=None, n=None, warnings=int(sum(stage0_warnings.values())), failures=0)

# The workflow after the replications (shared by Stage 0.5 and Stage 1)

The §1.5 metrics, the family-1 MCSE intervals, H19-Inf and H19-Cex (Holm 0.01 each).

Bootstrap call order (stream 4): family 1 only (`e12.family1_mcse`): the rows of the summary in order (cells in `e19.CELLS` order, then the estimators of the part), within a row CI length (mean), max weight (median; skipped where no weights are recorded), SE ratio; rows with fewer than two successes consume no draws.

In [3]:
def load_population():
    return json.loads((outputs.stage_dir(19, "stage0") / "stage0_e19_population.json").read_text())


def aggregate(raw, pop, seeds):
    summ = e19.summarise(raw, pop)
    summ = summ.assign(**e12.family1_mcse(raw, summ, seeds))
    v_inf, t_inf, u_inf = e19.family_inf(summ, raw)
    v_cex, t_cex, u_cex = e19.family_cex(summ, raw, pop)
    summ["family_inf"] = json.dumps(e19.family_sentence(v_inf, t_inf, u_inf))
    summ["family_inf_tests"] = json.dumps(t_inf)
    summ["family_inf_rejected"] = json.dumps(list(v_inf.rejected) if v_inf else [])
    summ["family_cex"] = json.dumps(e19.family_sentence(v_cex, t_cex, u_cex))
    summ["family_cex_tests"] = json.dumps(t_cex)
    summ["family_cex_rejected"] = json.dumps(list(v_cex.rejected) if v_cex else [])
    summ["unavailable_tests"] = json.dumps({"H19-Inf": u_inf, "H19-Cex": u_cex})
    return summ

# Stage 0.5: timing of the complete workflow, status counts, 1-versus-12-worker identity (no estimates kept)

Per cell and number of workers: the 10 replications and the aggregation of the cell. The aggregation of the whole pilot sample and the table and figure generation are timed once and recorded in the manifest. Pilot estimates, tables and figures are discarded.

In [4]:
if STAGE == "stage0.5":
    import matplotlib.pyplot as plt

    parallel.configure_worker()
    pop = load_population()
    timing, serial, para, all_tasks = [], [], [], []
    for workers in (1, outputs.PILOT_WORKERS):
        for ci, cell in enumerate(e19.CELLS):
            t0 = time.perf_counter()
            tasks = e19.tasks_for(PILOT_ENTROPY, outputs.PILOT_REPS, cells=[ci])
            res = parallel.run_tasks(e19.replicate, tasks, workers)
            aggregate(e19.raw_frame(tasks, res), pop, Seeds(19, entropy=PILOT_ENTROPY))
            timing.append({"cell": cell, "workers": workers, "seconds": time.perf_counter() - t0})
            if workers == 1:
                serial += res
                all_tasks += tasks
            else:
                para += res
    digest = parallel.assert_bit_identical(serial, para, expected=len(all_tasks))
    raw = e19.raw_frame(all_tasks, serial)
    t0 = time.perf_counter()
    buf = io.StringIO()
    aggregate(raw, pop, Seeds(19, entropy=PILOT_ENTROPY)).to_csv(buf, index=False)
    buf.seek(0)
    S = pd.read_csv(buf)
    e19.tables(S)
    plt.close(e19.figure(S))
    whole_seconds = time.perf_counter() - t0
    counts = raw.groupby(["cell", "arm", "status"]).size().reset_index(name="count")
    status_df = pd.DataFrame({"cell": [e19.CELLS[c] for c in counts["cell"]],
                              "arm": counts["arm"].str.replace("|", "-", regex=False),
                              "status": counts["status"], "count": counts["count"].astype(int)})
    rec.write_pilot("timing.csv", pd.DataFrame(timing))
    rec.write_pilot("status_counts.csv", status_df)
    rec.record_parallel_identity(cells=e19.CELLS, reps=outputs.PILOT_REPS, n_jobs=outputs.PILOT_WORKERS,
                                 digest=digest)
    print(pd.DataFrame(timing).groupby("workers")["seconds"].sum(), whole_seconds)
    print(status_df.groupby(["arm", "status"])["count"].sum().to_string())
    rec.finalize(R={p: outputs.PILOT_REPS for p in e19.PARTS}, n={p: list(e19.N_PART[p]) for p in e19.PARTS},
                 warnings=e19.total_warnings(raw), failures=int((raw["status"] != "ok").sum()),
                 extra={"whole_pilot_aggregation_table_and_figure_seconds": whole_seconds})
    del raw

# Stage 1: Monte Carlo (6 workers; results do not depend on the number of workers, Stage 0.5)

In [5]:
if STAGE == "stage1":
    parallel.configure_worker()
    tasks = e19.tasks_for(CONFIRMATORY_ENTROPY)
    t0 = time.perf_counter()
    results = parallel.run_tasks(e19.replicate, tasks, 6)
    print("seconds", time.perf_counter() - t0)
    raw = e19.raw_frame(tasks, results)
    rec.write_raw(raw)
    n_warn = e19.total_warnings(raw)
    raw.shape, n_warn

seconds 216.4803733749868


In [6]:
if STAGE == "stage1":
    summ = aggregate(raw, load_population(), Seeds(19))
    rec.write_summary(summ)
    {k: json.loads(summ[k].iloc[0]) for k in ("family_inf", "family_inf_rejected", "family_cex", "family_cex_rejected", "unavailable_tests")}

# Table and figure (from summary.csv only, §4)

In [7]:
if STAGE == "stage1":
    S = outputs.read_summary(rec)
    tabs, fams = e19.tables(S)
    for name, tex in tabs.items():
        rec.write_table(name, tex)
    rec.save_figure(e19.figure(S), "fig_E19_counterexamples")
    print(fams)
    rec.finalize(R=dict(e19.R_PART), n={p: list(e19.N_PART[p]) for p in e19.PARTS}, warnings=int(n_warn),
                 failures=int((S["R"] - S["R_s"]).sum()))

{'family_inf': 'H19-Inf: no departure from the prediction was detected', 'family_cex': 'H19-Cex: no departure from the prediction was detected'}
